# Exploratory analysis of the UCI Gene Expression Cancer RNA-Seq dataset

This notebook loads the gene expression matrix and labels, validates the dataset structure, checks for missing values, explores tumour class distribution and applies unsupervised and supervised machine learning methods.

In [ ]:
import os
import time

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from pathlib import Path

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.svm import LinearSVC

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    adjusted_rand_score,
    normalized_mutual_info_score
)

from sklearn.feature_selection import SelectKBest, f_classif, VarianceThreshold

os.makedirs("../output/figures", exist_ok=True)
os.makedirs("../output/tables", exist_ok=True)


## 2. Data loading

The two main dataset files are loaded: the expression matrix (`data.csv`) and the file containing tumour class labels (`labels.csv`).

In [ ]:
data = pd.read_csv("../input/data.csv", index_col=0)
labels = pd.read_csv("../input/labels.csv", index_col=0)

## 3. Initial inspection of the gene expression matrix

The expression matrix is inspected to confirm its structure. Each row corresponds to a sample and each column corresponds to a gene expression feature.

In [ ]:
data.head()

## 4. Initial inspection of the labels

The labels file is inspected to confirm that each sample has an associated tumour class.

In [ ]:
labels.head()

## 5. Data dimensions

The number of samples and expression features is confirmed before the following analyses.

In [ ]:
print("Expression matrix shape:", data.shape)
print("Labels shape:", labels.shape)

## 6. Alignment check between expression matrix and labels

Before applying any model, the sample order in the expression matrix and labels file is checked to ensure that each expression profile is associated with the correct tumour class.

In [ ]:
data.index.equals(labels.index)

## 7. Tumour class distribution

The number of samples per tumour class is counted and visualized. This step helps identify class imbalance before model evaluation.

In [ ]:
class_counts = labels.iloc[:, 0].value_counts()
class_counts

In [ ]:
colors = ["#0072B2", "#009E73", "#D55E00", "#CC79A7", "#F0E442"]

ax = class_counts.plot(kind="bar", color=colors, edgecolor="black", figsize=(7, 4))

plt.title("Tumour class distribution")
plt.xlabel("Tumour class")
plt.ylabel("Number of samples")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()

plt.savefig("../output/figures/class_distribution.png", dpi=300, bbox_inches="tight")

plt.show()

## 8. Missing value check

Missing values are checked in both the expression matrix and the labels file. This step is essential before applying dimensionality reduction or machine learning models.

In [ ]:
missing_data = data.isna().sum().sum()
missing_labels = labels.isna().sum().sum()

print("Missing values in expression matrix:", missing_data)
print("Missing values in labels:", missing_labels)

## 9. Data type check

The feature data types are verified to confirm that the expression matrix is numeric and suitable for PCA and machine learning models.

In [ ]:
data.dtypes.value_counts()

## 10. Feature name inspection

The first feature names are inspected to determine whether they correspond to directly interpretable biological identifiers or generic feature names.

In [ ]:
data.columns[:20]

## 11. Descriptive statistics of the expression matrix

Descriptive statistics are calculated to understand the scale and distribution of the expression values.

In [ ]:
data.describe()

## 12. Data standardization

Before applying PCA, the data are standardized. Standardization transforms each feature to zero mean and unit variance, preventing features with larger numeric scales from dominating the analysis.

In [ ]:
scaler = StandardScaler()
data_scaled = scaler.fit_transform(data)

data_scaled.shape

## 13. PCA for visualization of expression profiles

PCA is applied to reduce the expression matrix to two principal components. This allows visualization of whether samples from different tumour types show natural separation based on expression profiles.

In [ ]:
pca = PCA(n_components=2)
pca_result = pca.fit_transform(data_scaled)

pca_df = pd.DataFrame(
    pca_result,
    columns=["PC1", "PC2"],
    index=data.index
)

pca_df["Class"] = labels["Class"]

pca_df.head()

## 14. Explained variance of the principal components

Explained variance indicates the proportion of total variability captured by each principal component.

In [ ]:
explained_variance = pca.explained_variance_ratio_

print("Variância explicada pela PC1:", explained_variance[0])
print("Variância explicada pela PC2:", explained_variance[1])
print("Variância explicada total:", explained_variance.sum())

## 15. 2D PCA visualization

The scatter plot using the first two principal components allows partial separation between tumour classes to be inspected visually.

In [ ]:
colors = {"BRCA": "#0072B2", "KIRC": "#009E73", "LUAD": "#D55E00", "PRAD": "#CC79A7", "COAD": "#F0E442"}

plt.figure(figsize=(7, 5))

for cancer_type, color in colors.items():
    subset = pca_df[pca_df["Class"] == cancer_type]
    plt.scatter(
        subset["PC1"],
        subset["PC2"],
        label=cancer_type,
        alpha=0.7,
        s=35,
        color=color
    )

plt.title("PCA of gene expression profiles")
plt.xlabel(f"PC1 ({explained_variance[0]*100:.2f}% explained variance)")
plt.ylabel(f"PC2 ({explained_variance[1]*100:.2f}% explained variance)")
plt.legend(title="Tumour class")
plt.tight_layout()

plt.savefig("../output/figures/pca_2d.png", dpi=300, bbox_inches="tight")

plt.show()

The PCA visualization shows partial separation between tumour classes. KIRC appears more clearly separated from the remaining classes, while other tumour types show some overlap. This supports the use of PCA as an exploratory visualization tool, but also shows that two components are not enough to fully describe the structure of the dataset.

## Definition of variables for the analyses

Before the unsupervised and supervised analyses, the expression matrix is defined as `X` and the true tumour classes as `y`. The labels are not used to train K-Means; they are used only afterwards to evaluate the correspondence between clusters and real classes.

In [ ]:
X = data
y = labels["Class"]

## 16. Unsupervised model: PCA + K-Means

An unsupervised analysis with PCA and K-Means was performed before supervised classification. In this approach, the true labels are not used during clustering. The objective is to evaluate whether the gene expression profiles contain enough structure to form clusters associated with tumour types.

Different numbers of principal components were tested before K-Means. The best configuration was selected using ARI and NMI, after comparing the clusters with the real classes.

### 16.1 Evaluation of different numbers of PCA components

Because K-Means is sensitive to dimensionality, several PCA configurations were tested before clustering. Each configuration applies standardization, dimensionality reduction and K-Means with five clusters, corresponding to the number of tumour classes in the dataset.

The real labels are not used by the algorithm during clustering. They are used only afterwards to evaluate whether the discovered clusters correspond to known tumour classes.

In [ ]:
pca_components_list = [10, 20, 50, 100]

clustering_results = []

for n_components in pca_components_list:
    
    clustering_pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("pca", PCA(
            n_components=n_components,
            svd_solver="randomized",
            random_state=42
        )),
        ("kmeans", KMeans(
            n_clusters=5,
            random_state=42,
            n_init=20
        ))
    ])
    
    clusters = clustering_pipeline.fit_predict(X)
    
    ari = adjusted_rand_score(y, clusters)
    nmi = normalized_mutual_info_score(y, clusters)
    
    clustering_results.append({
        "PCA components": n_components,
        "ARI": ari,
        "NMI": nmi
    })

clustering_results_df = pd.DataFrame(clustering_results)
clustering_results_df

In [ ]:
clustering_results_df.to_csv(
    "../output/tables/kmeans_pca_comparison.csv",
    index=False
)

### 16.2 Selection of the best unsupervised configuration

The best configuration was selected based on the highest ARI value. ARI was used because it compares clustering results with real classes while correcting for agreement expected by chance.

In [ ]:
best_clustering_config = clustering_results_df.sort_values(
    by="ARI",
    ascending=False
).iloc[0]

best_pca_components = int(best_clustering_config["PCA components"])

print("Best number of PCA components:", best_pca_components)
print("ARI:", best_clustering_config["ARI"])
print("NMI:", best_clustering_config["NMI"])

### 16.3 Training the final unsupervised model

After selecting the best number of PCA components, the final unsupervised model was trained. This model combines standardization, PCA and K-Means in a single pipeline.

Unlike supervised models, this model does not directly predict tumour classes such as BRCA, KIRC, COAD, LUAD or PRAD. Instead, it assigns each sample to a numerical cluster.

In [ ]:
final_clustering_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(
        n_components=best_pca_components,
        svd_solver="randomized",
        random_state=42
    )),
    ("kmeans", KMeans(
        n_clusters=5,
        random_state=42,
        n_init=20
    ))
])

final_clusters = final_clustering_pipeline.fit_predict(data)


In [ ]:
joblib.dump(
    final_clustering_pipeline,
    "../output/final_model_kmeans_pca.joblib"
)

### 16.4 Saving cluster assignments for each sample

After training the final model, a table was created with the real class of each sample and the cluster assigned by K-Means. This table allows the relationship between discovered clusters and real tumour types to be analysed.

In [ ]:
final_cluster_assignments = pd.DataFrame({
    "Sample": data.index,
    "True class": y.values,
    "Cluster": final_clusters
})

final_cluster_assignments.to_csv(
    "../output/tables/final_kmeans_cluster_assignments.csv",
    index=False
)

final_cluster_assignments.head()

### 16.5 Correspondence between clusters and true classes

To interpret the clusters, a contingency table was created between K-Means clusters and real tumour classes. This analysis identifies whether specific clusters are dominated by particular tumour types.

In [ ]:
final_cluster_class_table = pd.crosstab(
    final_cluster_assignments["Cluster"],
    final_cluster_assignments["True class"]
)

final_cluster_class_table.to_csv(
    "../output/tables/final_kmeans_cluster_class_table.csv"
)

final_cluster_class_table

### 16.6 Final metrics of the unsupervised model

The final PCA + K-Means metrics were saved in a table. ARI and NMI quantify the agreement between clusters and true tumour classes, although the classes were not used during clustering.

In [ ]:
final_clustering_metrics = pd.DataFrame([{
    "Best PCA components": best_pca_components,
    "ARI": best_clustering_config["ARI"],
    "NMI": best_clustering_config["NMI"],
    "Number of clusters": 5
}])

final_clustering_metrics.to_csv(
    "../output/tables/final_kmeans_clustering_metrics.csv",
    index=False
)

final_clustering_metrics

### 16.7 Impact of PCA components on clustering

A plot was created to visualize how the number of PCA components influences K-Means performance. This analysis helps determine whether retaining more dimensional information improves the correspondence between clusters and real classes.

In [ ]:
plt.figure(figsize=(7, 4))

plt.plot(
    clustering_results_df["PCA components"],
    clustering_results_df["ARI"],
    marker="o"
)

plt.title("Impact of PCA components on K-Means clustering")
plt.xlabel("Number of PCA components")
plt.ylabel("Adjusted Rand Index (ARI)")
plt.xticks(clustering_results_df["PCA components"])
plt.tight_layout()

plt.savefig("../output/figures/kmeans_pca_ari_comparison.png", dpi=300, bbox_inches="tight")

plt.show()

### 16.8 Visualization of final clusters in 2D PCA space

To visualize the clusters assigned by the final model, samples were projected onto the first two principal components. The plot helps inspect spatial separation and possible overlap between groups.

In [ ]:
data_scaled_for_plot = StandardScaler().fit_transform(data)

pca_2d = PCA(n_components=2)
data_pca_2d = pca_2d.fit_transform(data_scaled_for_plot)

cluster_plot_df = pd.DataFrame(
    data_pca_2d,
    columns=["PC1", "PC2"],
    index=data.index
)

cluster_plot_df["Cluster"] = final_clusters
cluster_plot_df["Class"] = y.values

plt.figure(figsize=(7, 5))

for cluster_id in sorted(cluster_plot_df["Cluster"].unique()):
    subset = cluster_plot_df[cluster_plot_df["Cluster"] == cluster_id]
    plt.scatter(
        subset["PC1"],
        subset["PC2"],
        label=f"Cluster {cluster_id}",
        alpha=0.7,
        s=35
    )

plt.title("Clusters finais K-Means visualizados em PCA 2D")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend(title="Cluster")
plt.tight_layout()

plt.savefig("../output/figures/final_kmeans_clusters_pca_2d.png", dpi=300, bbox_inches="tight")

plt.show()

The final unsupervised model was selected by comparing different numbers of PCA components before K-Means. The labels were not used during clustering; they were only used afterwards to evaluate the agreement between clusters and true tumour classes using ARI and NMI.

## 17. Train/test split

To evaluate supervised models fairly, the data were split into training and test sets. The split was stratified to preserve tumour class proportions in both subsets.

In [ ]:
X = data
y = labels["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training set shape:", X_train.shape)
print("Test set shape:", X_test.shape)
print("\nClass distribution in training set:")
print(y_train.value_counts())
print("\nClass distribution in test set:")
print(y_test.value_counts())

## 18. Baseline model: Multiclass Logistic Regression

Logistic Regression was used as a baseline model for multiclass tumour classification. Despite its name, this algorithm is a supervised classification method. In this context, it learns a relationship between gene expression values and the corresponding tumour class.

In [ ]:
logreg_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=5000,
        solver="lbfgs",
        random_state=42
    ))
])

logreg_pipeline.fit(X_train, y_train)

y_pred_logreg = logreg_pipeline.predict(X_test)

## 19. Logistic Regression evaluation

The model was evaluated on the test set using accuracy, macro precision, macro recall and macro F1-score. Macro-averaged metrics are important because they calculate the average performance across classes, reducing the influence of class imbalance.

In [ ]:
accuracy = accuracy_score(y_test, y_pred_logreg)
precision_macro = precision_score(y_test, y_pred_logreg, average="macro")
recall_macro = recall_score(y_test, y_pred_logreg, average="macro")
f1_macro = f1_score(y_test, y_pred_logreg, average="macro")

print("Accuracy:", accuracy)
print("Precision macro:", precision_macro)
print("Recall macro:", recall_macro)
print("F1-score macro:", f1_macro)

print("\nClassification report:")
print(classification_report(y_test, y_pred_logreg))

logreg_results = pd.DataFrame({
    "Model": ["Logistic Regression"],
    "Accuracy": [accuracy],
    "Precision macro": [precision_macro],
    "Recall macro": [recall_macro],
    "F1-score macro": [f1_macro]
})

logreg_results.to_csv(
    "../output/tables/logistic_regression_results.csv",
    index=False
)

logreg_report = pd.DataFrame(
    classification_report(y_test, y_pred_logreg, output_dict=True)
).transpose()

logreg_report.to_csv(
    "../output/tables/logistic_regression_classification_report.csv",
    index=True
)

logreg_results


## 20. Logistic Regression confusion matrix

The confusion matrix shows, for each tumour class, how many samples were correctly classified and how many were confused with other classes.

In [ ]:
cm = confusion_matrix(y_test, y_pred_logreg, labels=logreg_pipeline.classes_)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=logreg_pipeline.classes_
)

fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, cmap="Blues", colorbar=True)

plt.title("Confusion matrix - Logistic Regression")
plt.tight_layout()

plt.savefig("../output/figures/confusion_matrix_logreg.png", dpi=300, bbox_inches="tight")

plt.show()

The Logistic Regression confusion matrix shows very high performance on the test set. Most classes were correctly classified, which indicates that the expression profiles contain strong discriminative signal even for a simple linear baseline model.

## 21. Linear SVM model

The second model evaluated was a Support Vector Machine with a linear kernel. This model is suitable for high-dimensional data such as RNA-Seq expression profiles because it learns linear decision boundaries in the feature space.

In [ ]:
svm_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", SVC(
        kernel="linear",
        random_state=42
    ))
])

svm_pipeline.fit(X_train, y_train)

y_pred_svm = svm_pipeline.predict(X_test)

## 22. Linear SVM evaluation

The Linear SVM was evaluated using the same metrics as Logistic Regression, allowing direct comparison between supervised models.

In [ ]:
accuracy_svm = accuracy_score(y_test, y_pred_svm)
precision_macro_svm = precision_score(y_test, y_pred_svm, average="macro")
recall_macro_svm = recall_score(y_test, y_pred_svm, average="macro")
f1_macro_svm = f1_score(y_test, y_pred_svm, average="macro")

print("Accuracy:", accuracy_svm)
print("Precision macro:", precision_macro_svm)
print("Recall macro:", recall_macro_svm)
print("F1-score macro:", f1_macro_svm)

print("\nClassification report:")
print(classification_report(y_test, y_pred_svm))

svm_results = pd.DataFrame({
    "Model": ["Linear SVM"],
    "Accuracy": [accuracy_svm],
    "Precision macro": [precision_macro_svm],
    "Recall macro": [recall_macro_svm],
    "F1-score macro": [f1_macro_svm]
})

svm_results.to_csv(
    "../output/tables/linear_svm_results.csv",
    index=False
)

svm_report = pd.DataFrame(
    classification_report(y_test, y_pred_svm, output_dict=True)
).transpose()

svm_report.to_csv(
    "../output/tables/linear_svm_classification_report.csv",
    index=True
)

svm_results


## 23. Linear SVM confusion matrix

The confusion matrix was used to inspect class-specific errors made by the Linear SVM model.

In [ ]:
cm_svm = confusion_matrix(y_test, y_pred_svm, labels=svm_pipeline.classes_)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_svm,
    display_labels=svm_pipeline.classes_
)

fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, cmap="Purples", colorbar=True)

plt.title("Confusion matrix - Linear SVM")
plt.tight_layout()

plt.savefig("../output/figures/confusion_matrix_svm.png", dpi=300, bbox_inches="tight")

plt.show()

Linear SVM showed the strongest performance at this stage, with high accuracy and macro F1-score. The confusion matrix shows only a very small number of errors, supporting the suitability of a linear classifier for this high-dimensional dataset.

## 24. k-NN model with PCA

k-NN is a distance-based classifier that assigns a class according to the nearest training samples in feature space. Because k-NN depends on distance calculations, the high dimensionality of RNA-Seq data can affect both performance and computational cost. For this reason, k-NN was combined with PCA to reduce the number of variables before classification.

## 25. Impact of the number of PCA components on k-NN

Different numbers of principal components were tested before k-NN. This analysis evaluates the balance between predictive performance and execution time.

Very aggressive dimensionality reduction may remove relevant discriminative information, while too many components can increase computational cost.

In [ ]:
pca_components_list = [10, 20, 30, 40, 50]

knn_pca_results = []

for n_components in pca_components_list:
    start_time = time.time()
    
    knn_pipeline_temp = Pipeline([
        ("scaler", StandardScaler()),
        ("pca", PCA(
            n_components=n_components,
            svd_solver="randomized",
            random_state=42
        )),
        ("model", KNeighborsClassifier(
            n_neighbors=5,
            n_jobs=-1
        ))
    ])
    
    knn_pipeline_temp.fit(X_train, y_train)
    y_pred_temp = knn_pipeline_temp.predict(X_test)
    
    elapsed_time = time.time() - start_time
    
    knn_pca_results.append({
        "PCA components": n_components,
        "Accuracy": accuracy_score(y_test, y_pred_temp),
        "Precision macro": precision_score(y_test, y_pred_temp, average="macro"),
        "Recall macro": recall_score(y_test, y_pred_temp, average="macro"),
        "F1-score macro": f1_score(y_test, y_pred_temp, average="macro"),
        "Time (seconds)": elapsed_time
    })

knn_pca_results_df = pd.DataFrame(knn_pca_results)
knn_pca_results_df

## 26. Export of k-NN with PCA results

The table containing the results obtained with different PCA component values was saved as a CSV file for later use in the report.

In [ ]:
knn_pca_results_df.to_csv(
    "../output/tables/knn_pca_components_comparison.csv",
    index=False
)

## 27. Impact of PCA components on k-NN performance

The following plot shows the variation of macro F1-score as a function of the number of principal components used before k-NN.

In [ ]:
plt.figure(figsize=(7, 4))

plt.plot(
    knn_pca_results_df["PCA components"],
    knn_pca_results_df["F1-score macro"],
    marker="o"
)

plt.title("Impact of the number of PCA components on k-NN")
plt.xlabel("Number of PCA components")
plt.ylabel("F1-score macro")
plt.xticks(knn_pca_results_df["PCA components"])
plt.tight_layout()

plt.savefig(
    "../output/figures/knn_pca_components_f1.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## 28. Impact of PCA components on execution time

In addition to predictive performance, execution time was evaluated for each configuration. This analysis supports discussion of the trade-off between performance and computational cost.

In [ ]:
plt.figure(figsize=(7, 4))

plt.plot(
    knn_pca_results_df["PCA components"],
    knn_pca_results_df["Time (seconds)"],
    marker="o"
)

plt.title("Execution time of k-NN with different PCA components")
plt.xlabel("Number of PCA components")
plt.ylabel("Execution time (seconds)")
plt.xticks(knn_pca_results_df["PCA components"])
plt.tight_layout()

plt.savefig(
    "../output/figures/knn_pca_components_time.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## 29. Selection of the final k-NN configuration

The final k-NN configuration was selected considering macro F1-score and information preservation after PCA. The configurations with 40 and 50 components showed equivalent performance on the test set. The 50-component configuration was retained because it preserves more information from the original space and showed stable execution in the tested environment.

In [ ]:
best_knn_config = knn_pca_results_df.sort_values(
    by="F1-score macro",
    ascending=False
).iloc[0]

best_n_components = int(best_knn_config["PCA components"])

print("Best number of PCA components:", best_n_components)
print("Best macro F1-score:", best_knn_config["F1-score macro"])
print("Execution time:", best_knn_config["Time (seconds)"])

## 30. Final k-NN model with PCA

After comparing different numbers of principal components, the final k-NN model was trained using the configuration with the best macro F1-score.

In [ ]:
knn_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(
        n_components=best_n_components,
        svd_solver="randomized",
        random_state=42
    )),
    ("model", KNeighborsClassifier(
        n_neighbors=5,
        n_jobs=-1
    ))
])

knn_pipeline.fit(X_train, y_train)

y_pred_knn = knn_pipeline.predict(X_test)

## 31. Evaluation of the final k-NN model with PCA

The final k-NN with PCA model was evaluated on the test set using the same metrics applied to the previous models.

In [ ]:
accuracy_knn = accuracy_score(y_test, y_pred_knn)
precision_macro_knn = precision_score(y_test, y_pred_knn, average="macro")
recall_macro_knn = recall_score(y_test, y_pred_knn, average="macro")
f1_macro_knn = f1_score(y_test, y_pred_knn, average="macro")

print("Accuracy:", accuracy_knn)
print("Precision macro:", precision_macro_knn)
print("Recall macro:", recall_macro_knn)
print("F1-score macro:", f1_macro_knn)

print("\nClassification report:")
print(classification_report(y_test, y_pred_knn))

knn_results = pd.DataFrame({
    "Model": ["k-NN + PCA"],
    "PCA components": [best_n_components],
    "Accuracy": [accuracy_knn],
    "Precision macro": [precision_macro_knn],
    "Recall macro": [recall_macro_knn],
    "F1-score macro": [f1_macro_knn]
})

knn_results.to_csv(
    "../output/tables/knn_pca_results.csv",
    index=False
)

knn_report = pd.DataFrame(
    classification_report(y_test, y_pred_knn, output_dict=True)
).transpose()

knn_report.to_csv(
    "../output/tables/knn_pca_classification_report.csv",
    index=True
)

knn_results

## 32. Confusion matrix of k-NN with PCA

The confusion matrix identifies correctly classified classes and the main errors made by k-NN after dimensionality reduction with PCA.

In [ ]:
cm_knn = confusion_matrix(
    y_test,
    y_pred_knn,
    labels=knn_pipeline.named_steps["model"].classes_
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_knn,
    display_labels=knn_pipeline.named_steps["model"].classes_
)

fig, ax = plt.subplots(figsize=(6, 5))

disp.plot(
    ax=ax,
    cmap="Greens",
    colorbar=True
)

plt.title("Confusion matrix - k-NN with PCA")
plt.tight_layout()

plt.savefig(
    "../output/figures/confusion_matrix_knn_pca.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## 33. Global model comparison

The three supervised models are compared: Logistic Regression, Linear SVM and k-NN with PCA. The comparison uses accuracy, macro precision, macro recall and macro F1-score.

In [ ]:
model_results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Linear SVM",
        "k-NN + PCA"
    ],
    "Accuracy": [
        accuracy,
        accuracy_svm,
        accuracy_knn
    ],
    "Precision macro": [
        precision_macro,
        precision_macro_svm,
        precision_macro_knn
    ],
    "Recall macro": [
        recall_macro,
        recall_macro_svm,
        recall_macro_knn
    ],
    "F1-score macro": [
        f1_macro,
        f1_macro_svm,
        f1_macro_knn
    ]
})

model_results.to_csv(
    "../output/tables/model_comparison.csv",
    index=False
)

model_results

## 34. Visual model comparison

Macro F1-score was used as the main metric for visual model comparison because it summarizes average class-level performance and is less dominated by class imbalance.

In [ ]:
plt.figure(figsize=(7, 4))

plt.bar(
    model_results["Model"],
    model_results["F1-score macro"],
    color=["#4C72B0", "#8172B3", "#55A868"],
    edgecolor="black"
)

plt.ylim(0.90, 1.00)
plt.title("Model comparison by macro F1-score")
plt.xlabel("Model")
plt.ylabel("F1-score macro")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()

plt.savefig(
    "../output/figures/model_comparison_f1.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## 35. Comparative interpretation of the models

Linear SVM achieved the best overall performance, with the highest macro F1-score among the evaluated models. Logistic Regression also achieved high performance, showing that linear relationships in gene expression profiles are already highly discriminative between tumour types.

k-NN with PCA performed well, although its results depended on the number of principal components used. The comparison across PCA configurations showed that dimensionality reduction is relevant for distance-based classification.

## 36. Stratified cross-validation of the models

Stratified cross-validation was applied to evaluate model stability. This approach splits the data into different folds while preserving tumour class proportions in each split. It allows assessment of whether the performance observed on the test set is maintained across multiple data partitions.

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "accuracy": "accuracy",
    "precision_macro": "precision_macro",
    "recall_macro": "recall_macro",
    "f1_macro": "f1_macro"
}

In [ ]:
logreg_cv_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=5000,
        solver="lbfgs",
        random_state=42
    ))
])

svm_cv_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", SVC(
        kernel="linear",
        random_state=42
    ))
])

knn_cv_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(
        n_components=50,
        svd_solver="randomized",
        random_state=42
    )),
    ("model", KNeighborsClassifier(
        n_neighbors=5,
        n_jobs=-1
    ))
])

In [ ]:
cv_models = {
    "Logistic Regression": logreg_cv_pipeline,
    "Linear SVM": svm_cv_pipeline,
    "k-NN + PCA": knn_cv_pipeline
}

cv_results = []

for model_name, model in cv_models.items():
    scores = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )
    
    cv_results.append({
        "Model": model_name,
        "Accuracy mean": scores["test_accuracy"].mean(),
        "Accuracy standard deviation": scores["test_accuracy"].std(),
        "Precision macro mean": scores["test_precision_macro"].mean(),
        "Recall macro mean": scores["test_recall_macro"].mean(),
        "F1-score macro mean": scores["test_f1_macro"].mean(),
        "F1-score macro standard deviation": scores["test_f1_macro"].std()
    })

cv_results_df = pd.DataFrame(cv_results)
cv_results_df.to_csv("../output/tables/cross_validation_results.csv", index=False)
cv_results_df

Stratified cross-validation evaluates model stability across different data splits. High mean scores with low standard deviations indicate that performance is not dependent on a single favourable train/test split.

Stratified cross-validation confirmed the stability of the models across different data splits. Linear SVM showed the strongest overall cross-validation performance, while Logistic Regression and k-NN with PCA also achieved high and stable scores.

## 37. Label-shuffling control

A lightweight label-shuffling control was performed to complement the supervised evaluation. The objective was to check whether the high performance observed with real labels would persist when the relationship between expression profiles and tumour classes was artificially disrupted.

This procedure was not treated as a formal permutation test because it was not used to estimate a robust p-value. It was used as a simple sanity check: if the model performs well with real labels but poorly with shuffled labels, this supports that it learned class-associated structure rather than random patterns.

In [ ]:
# Fast linear model for the label-shuffling control
label_shuffle_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LinearSVC(
        random_state=42,
        max_iter=5000
    ))
])

# Performance with real labels
label_shuffle_pipeline.fit(X_train, y_train)
real_pred = label_shuffle_pipeline.predict(X_test)

real_f1 = f1_score(
    y_test,
    real_pred,
    average="macro"
)

# Performance with shuffled training labels
n_shuffles = 5
shuffled_scores = []

rng = np.random.default_rng(42)

for shuffle_id in range(1, n_shuffles + 1):
    shuffled_y_train = pd.Series(
        rng.permutation(y_train.values),
        index=y_train.index
    )
    
    label_shuffle_pipeline.fit(X_train, shuffled_y_train)
    shuffled_pred = label_shuffle_pipeline.predict(X_test)
    
    shuffled_f1 = f1_score(
        y_test,
        shuffled_pred,
        average="macro"
    )
    
    shuffled_scores.append({
        "Shuffle": shuffle_id,
        "Shuffled F1-score macro": shuffled_f1
    })

label_shuffle_scores_df = pd.DataFrame(shuffled_scores)

label_shuffle_results = pd.DataFrame({
    "Real F1-score macro": [real_f1],
    "Mean shuffled F1-score macro": [label_shuffle_scores_df["Shuffled F1-score macro"].mean()],
    "Std shuffled F1-score macro": [label_shuffle_scores_df["Shuffled F1-score macro"].std()],
    "Number of shuffles": [n_shuffles]
})

label_shuffle_results.to_csv(
    "../output/tables/label_shuffle_control_results.csv",
    index=False
)

label_shuffle_scores_df.to_csv(
    "../output/tables/label_shuffle_control_scores.csv",
    index=False
)

label_shuffle_results

In [ ]:
plt.figure(figsize=(7, 4))

plt.hist(
    label_shuffle_scores_df["Shuffled F1-score macro"],
    bins=5,
    alpha=0.8
)

plt.axvline(
    real_f1,
    linestyle="--",
    linewidth=2,
    label="Real labels"
)

plt.title("Label-shuffling control - Linear SVM")
plt.xlabel("F1-score macro")
plt.ylabel("Frequency")
plt.legend()
plt.tight_layout()

plt.savefig(
    "../output/figures/label_shuffle_control_svm.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

The label-shuffling control compares the performance obtained with real labels against the performance obtained when training labels are randomly shuffled. This is a lightweight sanity check, not a formal permutation test. A strong decrease in performance after shuffling supports that the model learned class-associated structure rather than random patterns.

## 38. Feature selection with ANOVA F-test

Because of the high dimensionality of the dataset, feature selection was used to evaluate whether classification performance could be maintained using only a subset of the most discriminative features. `SelectKBest` with ANOVA F-test was applied inside a pipeline to avoid data leakage.

In [ ]:
feature_counts = [100, 500, 1000, 5000]

feature_selection_results = []

for k_features in feature_counts:
    fs_pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("variance", VarianceThreshold(threshold=0.0)),
        ("select", SelectKBest(
            score_func=f_classif,
            k=k_features
        )),
        ("model", SVC(
            kernel="linear",
            random_state=42
        ))
    ])
    
    fs_pipeline.fit(X_train, y_train)
    y_pred_fs = fs_pipeline.predict(X_test)
    
    feature_selection_results.append({
        "Number of features": k_features,
        "Accuracy": accuracy_score(y_test, y_pred_fs),
        "Precision macro": precision_score(y_test, y_pred_fs, average="macro"),
        "Recall macro": recall_score(y_test, y_pred_fs, average="macro"),
        "F1-score macro": f1_score(y_test, y_pred_fs, average="macro")
    })

feature_selection_results_df = pd.DataFrame(feature_selection_results)

feature_selection_results_df.to_csv(
    "../output/tables/feature_selection_results.csv",
    index=False
)

feature_selection_results_df

In [ ]:
plt.figure(figsize=(7, 4))

plt.plot(
    feature_selection_results_df["Number of features"],
    feature_selection_results_df["F1-score macro"],
    marker="o"
)

plt.title("Impact of feature selection on Linear SVM performance")
plt.xlabel("Number of selected features")
plt.ylabel("F1-score macro")
plt.xticks(feature_selection_results_df["Number of features"])
plt.tight_layout()

plt.savefig("../output/figures/feature_selection_f1.png", dpi=300, bbox_inches="tight")

plt.show()

Feature selection showed that Linear SVM maintained high performance even using only a small fraction of the 20,531 original features. This suggests that the discriminative signal is concentrated in a subset of expression variables.

## 39. Ranking of the most discriminative anonymous features

Although the dataset does not provide real gene symbols, Ensembl IDs or RefSeq IDs, it is possible to identify which anonymous features have higher discriminative power. A ranking of the top 100 features selected by ANOVA F-test was generated. These features are kept with their original identifiers (`gene_x`) to avoid incorrect biological interpretation.

In [ ]:
top_feature_pipeline = Pipeline([
    ("variance", VarianceThreshold(threshold=0.0)),
    ("select", SelectKBest(score_func=f_classif, k=100))
])

top_feature_pipeline.fit(X_train, y_train)

variance_step = top_feature_pipeline.named_steps["variance"]
select_step = top_feature_pipeline.named_steps["select"]

features_after_variance = X_train.columns[variance_step.get_support()]
selected_features = features_after_variance[select_step.get_support()]
selected_scores = select_step.scores_[select_step.get_support()]

top_features_df = pd.DataFrame({
    "Feature": selected_features,
    "ANOVA F-score": selected_scores
}).sort_values(by="ANOVA F-score", ascending=False)

top_features_df.to_csv(
    "../output/tables/top_selected_features.csv",
    index=False
)

top_features_df.head(20)

This ranking identifies expression features with higher discriminative power between tumour classes. However, because the dataset does not provide real gene symbols or biological identifiers, these features should be interpreted only as anonymous computational variables, not as specific genes or biomarkers.

## 40. Demonstration of single-sample classification

After model evaluation, Linear SVM was selected as the final supervised model because it achieved the best overall performance. This section demonstrates how the model can be used to predict the tumour class of one sample from the test set.

In [ ]:
tumor_info = {
    "BRCA": "Breast invasive carcinoma",
    "KIRC": "Kidney renal clear cell carcinoma",
    "COAD": "Colon adenocarcinoma",
    "LUAD": "Lung adenocarcinoma",
    "PRAD": "Prostate adenocarcinoma"
}

sample_id = X_test.index[0]

sample = X_test.loc[[sample_id]]
true_class = y_test.loc[sample_id]
predicted_class = svm_pipeline.predict(sample)[0]

print("Sample:", sample_id)
print("True class:", true_class, "-", tumor_info[true_class])
print("Predicted class:", predicted_class, "-", tumor_info[predicted_class])

if true_class == predicted_class:
    print("Result: correct classification")
else:
    print("Result: incorrect classification")

## 41. Export of the final model

The final model was saved to a file to allow future reuse with new samples that have the same feature format.

In [ ]:
joblib.dump(svm_pipeline, "../output/final_model_linear_svm.joblib")

loaded_model = joblib.load("../output/final_model_linear_svm.joblib")

test_prediction = loaded_model.predict(sample)[0]
print("Prediction with loaded model:", test_prediction)

This demonstration shows how the pipeline can be applied to an individual sample. In this example, the selected sample is classified with the exported Linear SVM model.

## 42. Computational complexity analysis

Computational complexity was summarized for the main pipeline steps. This is relevant because the dataset contains 801 samples and 20,531 expression features, which is a typical high-dimensional RNA-Seq setting.

The notation used is: `n` for number of samples, `p` for number of features, `r` for PCA components, `q` for clusters, `c` for classes, `i` for iterations and `F` for cross-validation folds.

In [ ]:
n_samples, n_features = data.shape
n_classes = y.nunique()
n_clusters = 5
n_folds = 5

try:
    selected_pca_components = best_pca_components
except NameError:
    selected_pca_components = 50

complexity_table = pd.DataFrame({
    "Step": [
        "Data validation",
        "Standardization",
        "PCA",
        "K-Means",
        "Logistic Regression",
        "Linear SVM",
        "k-NN training",
        "k-NN prediction",
        "SelectKBest ANOVA",
        "Cross-validation"
    ],
    "Approximate time complexity": [
        "O(n × p)",
        "O(n × p)",
        "O(n × p × r)",
        "O(i × n × q × r)",
        "O(i × n × p × c)",
        "Approximately O(n × p) to O(n² × p)",
        "O(n_train × r)",
        "O(n_test × n_train × r)",
        "O(n × p)",
        "O(F × model cost)"
    ],
    "Application in this project": [
        f"Validation of the matrix with {n_samples} samples and {n_features} features.",
        "Mean and standard deviation calculated for each feature.",
        f"Reduction from {n_features} features to {selected_pca_components} principal components.",
        f"Clustering of samples into {n_clusters} clusters in PCA space.",
        f"Multiclass supervised baseline with {n_classes} classes.",
        "Final supervised model for tumour classification.",
        "Stores training samples after PCA.",
        "Computes distances between test and training samples.",
        "Selects anonymous features with higher discriminative power.",
        f"Repeats training and evaluation across {n_folds} folds."
    ]
})

complexity_table.to_csv(
    "../output/tables/complexity_analysis.csv",
    index=False
)

complexity_table

The analysis shows that the main computational cost of the pipeline comes from the high dimensionality of the dataset (`p = 20531`). Standardization and feature selection scale with the expression matrix size, while PCA adds cost but reduces the feature space before K-Means and k-NN. Cross-validation increases execution time because training and evaluation are repeated several times.